# Mapa semántico de tesis del CIDE

Este cuaderno construye un mapa semántico de las tesis de Licenciatura en Economía del CIDE usando embeddings multilingües, UMAP, clustering, etiquetas automáticas de temas, evolución temporal y cruces con asesores homologados.

La entrada canónica es `tesis_lic_economia_cide.parquet`. No hace scraping: si quieres actualizar datos, corre primero `make scrape`.


## Dependencias y configuración

El modelo por defecto es `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`: es ligero para CPU/8 GB RAM, funciona bien en español e inglés y evita separar tesis solo por idioma. Puedes cambiarlo con la variable de entorno `ST_MODEL_NAME` sin editar la notebook.


In [ ]:
import os
import re
import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import umap
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

RANDOM_STATE = 420
DATA_PATH = Path('tesis_lic_economia_cide.parquet')
EMBEDDINGS_PATH = Path('embeddings_tesis.parquet')
CLUSTERS_PATH = Path('clusters_tesis.parquet')
CLUSTER_SUMMARY_PATH = Path('clusters_resumen.parquet')
CLUSTER_DIAGNOSTICS_PATH = Path('cluster_diagnostics.parquet')
CLUSTER_YEAR_PATH = Path('cluster_anio.parquet')
CLUSTER_LANGUAGE_PATH = Path('cluster_idioma.parquet')
ADVISOR_CLUSTER_PATH = Path('asesor_cluster_resumen.parquet')
ADVISOR_SUMMARY_PATH = Path('asesor_resumen.parquet')

MODEL_NAME = os.environ.get('ST_MODEL_NAME', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = int(os.environ.get('ST_BATCH_SIZE', '16'))

print(f'Modelo de embeddings: {MODEL_NAME}')
print(f'Dispositivo: {DEVICE}; batch_size={BATCH_SIZE}')


## Cargar y preparar los datos

Se combinan título y resumen. La columna `asesor_unificado` ya viene homologada desde el pipeline de scraping.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No encontré {DATA_PATH}. Corre primero ScrapingTesisLicEcoCIDE.qmd o make scrape para generar el Parquet canónico.'
    )

df_raw = pd.read_parquet(DATA_PATH)

required_cols = {'anio_pub', 'autorx', 'titulo', 'resumen', 'asesor_unificado', 'url'}
missing_cols = required_cols.difference(df_raw.columns)
if missing_cols:
    raise ValueError(f'Faltan columnas requeridas en {DATA_PATH}: {sorted(missing_cols)}')

df_raw['anio_pub'] = pd.to_numeric(df_raw['anio_pub'], errors='coerce').astype('Int64')

text_cols = ['titulo', 'resumen']
df = df_raw.dropna(subset=text_cols).copy()
df['texto'] = (df['titulo'].fillna('') + '. ' + df['resumen'].fillna('')).str.replace(r'\s+', ' ', regex=True).str.strip()
df = df[df['texto'].str.len() > 20].reset_index(drop=True)

df['asesor_unificado'] = df['asesor_unificado'].fillna('Asesor desconocido').replace('', 'Asesor desconocido')
df['text_hash'] = df['texto'].map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())

print(f'Tesis disponibles para analizar: {len(df)}')
print(f'Años: {df["anio_pub"].min()}-{df["anio_pub"].max()}')
print(f'Asesores homologados distintos: {df["asesor_unificado"].nunique()}')
display(df[['anio_pub', 'titulo', 'asesor_unificado', 'idioma', 'url']].head(3))


## Generar o cargar embeddings multilingües

Los embeddings se guardan en Parquet para no recalcularlos si el texto y el modelo no cambiaron. Esto permite iterar sobre UMAP, clustering y visualizaciones sin volver a descargar ni ejecutar el transformer.


In [ ]:
def embedding_columns(frame):
    return [c for c in frame.columns if c.startswith('embedding_')]

cache_ok = False
if EMBEDDINGS_PATH.exists():
    cached = pd.read_parquet(EMBEDDINGS_PATH)
    emb_cols = embedding_columns(cached)
    cache_ok = (
        len(cached) == len(df)
        and emb_cols
        and set(['url', 'text_hash', 'model_name']).issubset(cached.columns)
        and cached['url'].tolist() == df['url'].tolist()
        and cached['text_hash'].tolist() == df['text_hash'].tolist()
        and cached['model_name'].nunique() == 1
        and cached['model_name'].iloc[0] == MODEL_NAME
    )

if cache_ok:
    embeddings = cached[emb_cols].to_numpy(dtype=np.float32)
    print(f'Embeddings cargados desde cache: {EMBEDDINGS_PATH} {embeddings.shape}')
else:
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    embeddings = model.encode(
        df['texto'].tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    emb_cols = [f'embedding_{i:03d}' for i in range(embeddings.shape[1])]
    embeddings_export = pd.DataFrame(embeddings, columns=emb_cols)
    embeddings_export.insert(0, 'model_name', MODEL_NAME)
    embeddings_export.insert(0, 'text_hash', df['text_hash'])
    embeddings_export.insert(0, 'url', df['url'])
    embeddings_export.to_parquet(EMBEDDINGS_PATH, index=False)
    print(f'Embeddings generados y guardados: {EMBEDDINGS_PATH.resolve()} {embeddings.shape}')

if not np.all(np.isfinite(embeddings)):
    raise ValueError('Los embeddings contienen valores no finitos.')


## Reducir dimensionalidad con UMAP

UMAP se usa como layout visual. El clustering principal se calcula sobre embeddings originales para evitar que la geometría 2D determine artificialmente los grupos.


In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.05,
    metric='cosine',
    random_state=RANDOM_STATE,
)

umap_2d = umap_model.fit_transform(embeddings)

df_layout = df.copy()
df_layout['umap_x'] = umap_2d[:, 0]
df_layout['umap_y'] = umap_2d[:, 1]
display(df_layout[['titulo', 'anio_pub', 'umap_x', 'umap_y']].head(3))


## Diagnóstico de número de clusters

Se comparan K-Means sobre embeddings originales y sobre UMAP 2D para varios valores de `k`. La selección automática usa silhouette sobre embeddings, que es más cercana al espacio semántico real.


In [ ]:
def evaluate_kmeans(features, feature_space, k_values, metric_for_silhouette):
    rows = []
    for k in k_values:
        if k >= len(features):
            continue
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = km.fit_predict(features)
        rows.append({
            'model_name': MODEL_NAME,
            'feature_space': feature_space,
            'k': k,
            'silhouette': silhouette_score(features, labels, metric=metric_for_silhouette),
            'davies_bouldin': davies_bouldin_score(features, labels),
            'calinski_harabasz': calinski_harabasz_score(features, labels),
            'inertia': km.inertia_,
        })
    return pd.DataFrame(rows)

k_values = list(range(4, min(16, len(df_layout))))
diagnostics = pd.concat(
    [
        evaluate_kmeans(embeddings, 'embeddings', k_values, 'cosine'),
        evaluate_kmeans(umap_2d, 'umap_2d', k_values, 'euclidean'),
    ],
    ignore_index=True,
)

best_row = diagnostics[diagnostics['feature_space'] == 'embeddings'].sort_values(
    ['silhouette', 'davies_bouldin'], ascending=[False, True]
).iloc[0]
selected_k = int(best_row['k'])
diagnostics['selected'] = (diagnostics['feature_space'].eq('embeddings') & diagnostics['k'].eq(selected_k))
diagnostics['generated_at'] = datetime.now(timezone.utc).isoformat()
diagnostics.to_parquet(CLUSTER_DIAGNOSTICS_PATH, index=False)

print(f'k seleccionado: {selected_k}')
display(diagnostics.sort_values(['feature_space', 'k']))

fig = px.line(
    diagnostics,
    x='k',
    y='silhouette',
    color='feature_space',
    markers=True,
    title='Diagnóstico de clusters por silhouette',
    template='plotly_white',
    width=900,
    height=420,
)
fig.add_vline(x=selected_k, line_dash='dash', line_color='black')
fig.show()


## Clustering semántico sobre embeddings

El cluster final se calcula en el espacio de embeddings multilingües. UMAP queda como visualización.


In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=50)
cluster_labels = kmeans.fit_predict(embeddings)
cluster_algo = 'kmeans (embeddings)'

df_layout['cluster_id'] = cluster_labels
df_layout['cluster_label'] = df_layout['cluster_id'].map(lambda x: f'Cluster {x}')

print(f'Algoritmo de clustering activo: {cluster_algo} (k={selected_k})')

cluster_summary_base = (
    df_layout.groupby(['cluster_label', 'cluster_id'])
    .agg(
        conteo=('titulo', 'count'),
        anio_promedio=('anio_pub', 'mean'),
        anio_min=('anio_pub', 'min'),
        anio_max=('anio_pub', 'max'),
    )
    .reset_index()
    .sort_values('conteo', ascending=False)
)

display(cluster_summary_base)


## Etiquetar clusters con keywords y tesis representativas

Se usa TF-IDF por cluster para términos característicos y similitud al centroide para tesis representativas.


In [ ]:
SPANISH_STOPWORDS = {
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando',
    'de', 'del', 'desde', 'donde', 'dos', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'eran', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estado', 'estados',
    'estan', 'estar', 'estas', 'este', 'esto', 'estos', 'fue', 'fueron', 'ha', 'hacia', 'han', 'hasta',
    'hay', 'la', 'las', 'le', 'les', 'lo', 'los', 'mas', 'más', 'me', 'mediante', 'mi', 'mientras',
    'muy', 'no', 'nos', 'o', 'para', 'pero', 'por', 'porque', 'que', 'se', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'también', 'te', 'tiene', 'tienen', 'un', 'una', 'uno', 'unos', 'y', 'ya',
    'méxico', 'mexico', 'cide', 'tesis', 'licenciatura', 'economía', 'economia', 'análisis', 'analisis',
    'modelo', 'modelos', 'caso', 'estudio', 'evidencia', 'efecto', 'efectos', 'datos', 'resultados'
}
STOPWORDS = sorted(set(ENGLISH_STOP_WORDS).union(SPANISH_STOPWORDS))

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    max_features=5000,
)

tfidf = vectorizer.fit_transform(df_layout['texto'])
terms = np.array(vectorizer.get_feature_names_out())

keywords_by_cluster = {}
for cid in sorted(df_layout['cluster_id'].unique()):
    mask = df_layout['cluster_id'].to_numpy() == cid
    scores = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top_idx = scores.argsort()[::-1][:10]
    keywords_by_cluster[cid] = ', '.join(terms[top_idx])

embeddings_norm = normalize(embeddings)
centroids_norm = normalize(kmeans.cluster_centers_)
representatives = {}
for cid in sorted(df_layout['cluster_id'].unique()):
    mask_idx = np.where(df_layout['cluster_id'].to_numpy() == cid)[0]
    sims = cosine_similarity(embeddings_norm[mask_idx], centroids_norm[[cid]]).ravel()
    top_local = mask_idx[np.argsort(sims)[::-1][:3]]
    representatives[cid] = ' | '.join(df_layout.loc[top_local, 'titulo'].tolist())

top_advisors = (
    df_layout.groupby(['cluster_id', 'asesor_unificado'])
    .size()
    .reset_index(name='n')
    .sort_values(['cluster_id', 'n'], ascending=[True, False])
    .groupby('cluster_id')
    .head(5)
    .groupby('cluster_id')
    .apply(lambda g: ', '.join(f'{r.asesor_unificado} ({r.n})' for r in g.itertuples()), include_groups=False)
    .to_dict()
)

cluster_overview = cluster_summary_base.copy()
cluster_overview['keywords'] = cluster_overview['cluster_id'].map(keywords_by_cluster)
cluster_overview['tesis_representativas'] = cluster_overview['cluster_id'].map(representatives)
cluster_overview['asesores_top'] = cluster_overview['cluster_id'].map(top_advisors)
cluster_overview['cluster_algo'] = cluster_algo
cluster_overview['embedding_model'] = MODEL_NAME
cluster_overview = cluster_overview.sort_values('conteo', ascending=False)

display(cluster_overview[['cluster_label', 'conteo', 'anio_promedio', 'keywords', 'tesis_representativas']])


## Mapa semántico interactivo


In [ ]:
hover_cols = {
    'titulo': True,
    'autorx': True,
    'anio_pub': True,
    'asesor_unificado': True,
    'url': True,
    'cluster_label': False,
}

fig = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_label',
    hover_name='titulo',
    hover_data=hover_cols,
    title='Mapa semántico de tesis de Lic. en Economía (CIDE)',
    template='plotly_white',
    width=950,
    height=650,
)
fig.update_traces(marker=dict(size=8, opacity=0.82, line=dict(width=0)))
fig.update_layout(legend_title_text='Cluster')
fig.show()


## Evolución temporal de temas

Esta tabla y heatmap muestran qué clusters crecen o pierden peso por año.


In [ ]:
cluster_year = (
    df_layout.dropna(subset=['anio_pub'])
    .groupby(['anio_pub', 'cluster_label', 'cluster_id'])
    .size()
    .reset_index(name='conteo')
)
cluster_year['total_anio'] = cluster_year.groupby('anio_pub')['conteo'].transform('sum')
cluster_year['participacion_anio'] = cluster_year['conteo'] / cluster_year['total_anio']
cluster_year.to_parquet(CLUSTER_YEAR_PATH, index=False)

heatmap_data = cluster_year.pivot_table(
    index='cluster_label',
    columns='anio_pub',
    values='participacion_anio',
    fill_value=0,
)

fig = px.imshow(
    heatmap_data,
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Participación de cada cluster por año',
    labels=dict(x='Año', y='Cluster', color='Participación'),
    width=980,
    height=520,
)
fig.show()

display(cluster_year.sort_values(['anio_pub', 'cluster_id']).head(20))


## Distribución de idioma por cluster

Esta tabla verifica si el modelo multilingüe está separando tesis por tema en vez de aislarlas por idioma. Si las tesis en inglés aparecen repartidas entre varios clusters, el espacio semántico está funcionando mejor para análisis bilingüe.


In [ ]:

if 'idioma' in df_layout.columns:
    cluster_language = (
        df_layout.assign(idioma=df_layout['idioma'].fillna('desconocido'))
        .groupby(['cluster_label', 'cluster_id', 'idioma'])
        .size()
        .reset_index(name='conteo')
    )
    cluster_language['total_cluster'] = cluster_language.groupby('cluster_label')['conteo'].transform('sum')
    cluster_language['participacion_cluster'] = cluster_language['conteo'] / cluster_language['total_cluster']
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)

    display(cluster_language.sort_values(['cluster_id', 'idioma']))

    fig = px.bar(
        cluster_language,
        x='cluster_label',
        y='conteo',
        color='idioma',
        title='Distribución de idioma por cluster',
        template='plotly_white',
        width=850,
        height=430,
    )
    fig.show()
else:
    cluster_language = pd.DataFrame(columns=['cluster_label', 'cluster_id', 'idioma', 'conteo', 'total_cluster', 'participacion_cluster'])
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)
    print('No hay columna idioma para evaluar distribución por cluster.')


## Cruce asesores × temas

Ahora que los asesores están homologados, se puede medir concentración temática y diversidad por asesor.


In [ ]:
advisor_cluster = (
    df_layout.groupby(['asesor_unificado', 'cluster_label', 'cluster_id'])
    .agg(
        n_tesis=('titulo', 'count'),
        anio_min=('anio_pub', 'min'),
        anio_max=('anio_pub', 'max'),
    )
    .reset_index()
)
advisor_cluster['total_asesor'] = advisor_cluster.groupby('asesor_unificado')['n_tesis'].transform('sum')
advisor_cluster['participacion_asesor'] = advisor_cluster['n_tesis'] / advisor_cluster['total_asesor']
advisor_cluster = advisor_cluster.sort_values(['total_asesor', 'asesor_unificado', 'n_tesis'], ascending=[False, True, False])
advisor_cluster.to_parquet(ADVISOR_CLUSTER_PATH, index=False)

advisor_summary = (
    advisor_cluster.sort_values(['asesor_unificado', 'n_tesis'], ascending=[True, False])
    .groupby('asesor_unificado')
    .agg(
        n_tesis=('n_tesis', 'sum'),
        n_clusters=('cluster_id', 'nunique'),
        cluster_principal=('cluster_label', 'first'),
        participacion_cluster_principal=('participacion_asesor', 'first'),
        anio_min=('anio_min', 'min'),
        anio_max=('anio_max', 'max'),
    )
    .reset_index()
    .sort_values('n_tesis', ascending=False)
)
advisor_summary.to_parquet(ADVISOR_SUMMARY_PATH, index=False)

display(advisor_summary.head(20))

fig = px.scatter(
    advisor_summary.head(35),
    x='n_tesis',
    y='n_clusters',
    size='n_tesis',
    color='participacion_cluster_principal',
    hover_name='asesor_unificado',
    hover_data=['cluster_principal', 'anio_min', 'anio_max'],
    title='Volumen y diversidad temática por asesor/a',
    template='plotly_white',
    width=900,
    height=520,
    labels={
        'n_tesis': 'Tesis asesoradas',
        'n_clusters': 'Clusters distintos',
        'participacion_cluster_principal': 'Concentración en cluster principal',
    },
)
fig.show()


## Top asesores por cluster


In [ ]:
for cid in sorted(df_layout['cluster_id'].unique()):
    cluster_name = f'Cluster {cid}'
    subset = df_layout[df_layout['cluster_id'] == cid]
    top5 = (
        subset.groupby('asesor_unificado')
        .size()
        .reset_index(name='conteo')
        .sort_values('conteo', ascending=False)
        .head(5)
    )

    if top5.empty:
        continue

    fig = px.bar(
        top5,
        x='conteo',
        y='asesor_unificado',
        orientation='h',
        color='asesor_unificado',
        title=f'Top 5 asesores en {cluster_name}',
        text='conteo',
        color_discrete_sequence=px.colors.qualitative.Plotly,
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(
        template='plotly_white',
        height=320,
        xaxis_title='Número de tesis',
        yaxis_title='Asesor/a',
        yaxis=dict(categoryorder='array', categoryarray=top5['asesor_unificado'].tolist()[::-1]),
        showlegend=False,
        margin=dict(t=60, b=40, l=40, r=20),
    )
    fig.show()


## Exportar resultados analíticos


In [ ]:
cols_export = [
    'cluster_label',
    'cluster_id',
    'titulo',
    'resumen',
    'autorx',
    'anio_pub',
    'asesor_unificado',
    'idioma',
    'url',
]

clusters_export = (
    df_layout[cols_export + ['umap_x', 'umap_y']]
    .sort_values(['cluster_id', 'anio_pub', 'titulo'])
    .reset_index(drop=True)
)

clusters_export.to_parquet(CLUSTERS_PATH, index=False)
cluster_overview.to_parquet(CLUSTER_SUMMARY_PATH, index=False)

print(f'Archivo exportado: {CLUSTERS_PATH.resolve()}')
print(f'Resumen exportado: {CLUSTER_SUMMARY_PATH.resolve()}')
print(f'Diagnóstico exportado: {CLUSTER_DIAGNOSTICS_PATH.resolve()}')
print(f'Evolución temporal exportada: {CLUSTER_YEAR_PATH.resolve()}')
print(f'Distribución por idioma exportada: {CLUSTER_LANGUAGE_PATH.resolve()}')
print(f'Cruce asesor-cluster exportado: {ADVISOR_CLUSTER_PATH.resolve()}')
print(f'Resumen de asesores exportado: {ADVISOR_SUMMARY_PATH.resolve()}')


## Red de tesis por asesor

Esta red conecta tesis asesoradas por la misma persona. Es útil como vista institucional complementaria al mapa semántico.


In [ ]:
graph_df = df_layout.copy()

G = nx.Graph()
G.add_nodes_from(graph_df.index)

for advisor, idxs in graph_df.groupby('asesor_unificado').groups.items():
    ids = list(idxs)
    if len(ids) < 2:
        continue
    for i, j in combinations(ids, 2):
        w = G.get_edge_data(i, j, {}).get('weight', 0) + 1
        G.add_edge(i, j, weight=w, advisor=advisor)

if G.number_of_edges() == 0:
    print('No hay conexiones por asesor (todos únicos o faltantes).')
    df_layout['advisor_comm_id'] = -1
    df_layout['advisor_comm_label'] = 'Sin comunidad'
else:
    communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    comm_map = {node: cid for cid, nodes in enumerate(communities) for node in nodes}
    df_layout['advisor_comm_id'] = df_layout.index.map(lambda x: comm_map.get(x, -1))
    df_layout['advisor_comm_label'] = df_layout['advisor_comm_id'].map(lambda x: f'AdvisorComm {x}' if x != -1 else 'Sin comunidad')

    comm_summary = (
        df_layout.groupby('advisor_comm_label')
        .agg(
            conteo=('titulo', 'count'),
            asesores_top=('asesor_unificado', lambda s: ', '.join(s.value_counts().head(3).index)),
        )
        .sort_values('conteo', ascending=False)
    )
    print(f'Comunidades detectadas: {len(communities)}')
    display(comm_summary.head(20))


## Visualizar red de asesores


In [ ]:
if 'advisor_comm_label' not in df_layout.columns or G.number_of_edges() == 0:
    print('No hay red de asesores para visualizar.')
else:
    pos = nx.spring_layout(G, weight='weight', seed=42)

    nodes_df = (
        df_layout[['titulo', 'asesor_unificado', 'cluster_label', 'advisor_comm_label']]
        .assign(
            x=lambda d: d.index.map(lambda i: pos[i][0]),
            y=lambda d: d.index.map(lambda i: pos[i][1]),
        )
    )

    communities = sorted(nodes_df['advisor_comm_label'].unique())
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Safe + px.colors.qualitative.Set3
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(communities)}
    nodes_df['color'] = nodes_df['advisor_comm_label'].map(color_map)

    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=0.5, color='#999'),
        hoverinfo='none',
        mode='lines',
    )

    node_trace = go.Scatter(
        x=nodes_df['x'],
        y=nodes_df['y'],
        mode='markers',
        marker=dict(size=8, color=nodes_df['color'], line=dict(width=0.5, color='#333')),
        text=nodes_df.apply(
            lambda r: f"{r['titulo']}<br>Asesor: {r['asesor_unificado']}<br>Comunidad: {r['advisor_comm_label']}<br>Cluster semántico: {r['cluster_label']}",
            axis=1,
        ),
        hoverinfo='text',
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title='Red de tesis conectadas por asesor (comunidades por modularidad)',
        template='plotly_white',
        showlegend=False,
        width=950,
        height=700,
        margin=dict(l=30, r=30, t=60, b=30),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )
    fig.show()
